# 单元09：学习曲线与K折交叉验证

状态：学习中，2026-10-29建立。本Notebook是逐步填写的教学工作簿，不是已完成实验。

材料情境为7085Al/ZrB2，所有新数据必须标注synthetic data。2026-10-30教师按预设配方生成150条教学CSV；学生已提示后完成120条开发/30条保留测试的首次划分并保存shape输出，尚未拟合模型，没有真实模型曲线、指标或正式评估结果。

课程依据：markdown/week6.md第10.3—10.6节；code/ex5-bias vs variance/。沿用单元08已经学过的线性回归、多项式和Ridge，本单元不引入其他模型。


## 实验预设（在数据生成和查看结果前固定）

本单元只使用全新的 **synthetic data（人工模拟数据）**，不复用单元08的CSV、训练/验证/测试分组或已知分数。以下设置是在生成数据前确定的实验设计；2026-10-30已完成CSV准备与学生120/30首次划分，后续96/24、5折和模型评价仍未执行，不把计划写成实际结果。

- 教学问题：单一输入 ZrB2_wt_pct → 模拟目标 UTS_MPa。wt.%数值1表示1 wt.%，UTS单位MPa；关系为数学教学设定，不是7085Al强化机制或实验标定。
- 新数据配方：NumPy default_rng(20261029)；先生成150个Uniform(0,2)输入，再生成150个Normal(0,6)噪声；模拟目标为465 + 60*x - 24*x**2 + noise。不按预期图形重抽样或更换种子。
- 以sample_id固定追溯行身份。原始输入X预期shape=(150,1)，目标y沿用二维列约定(150,1)；这些是计划shape，运行后另核验。
- 在任何拟合、选模、目标散点图或指标分析之前，用train_test_split(test_size=0.2, random_state=20261030)分离120条开发数据、30条独立保留测试数据。记录索引；保留测试不预览目标、作图、参与曲线或交叉验证。
- 先在开发数据内部按test_size=0.2、random_state=20261031建立固定96训练/24验证的入门学习曲线。训练子集顺序按固定划分返回顺序保留；样本量预设为12、24、48、72、96。每点重新训练完整Pipeline，训练误差仅在对应训练子集上计算，验证组固定。有限曲线无需严格单调。
- 然后在全部120条开发数据内部用KFold(n_splits=5, shuffle=True, random_state=20261032)，两个候选共用同一组折索引。每折计划96训练/24验证；只用开发数据，不把保留测试混入K-fold。
- 仅比较两个预设候选：StandardScaler → LinearRegression（一次线性回归）；PolynomialFeatures(degree=4, include_bias=False) → StandardScaler → Ridge(alpha=1.0)。次数和强度是在查看结果前给定的教学方案，不由单元08最优值迁移，不开展网格搜索或新模型。
- 每个训练子集和每个训练折都拟合自己的完整Pipeline。PolynomialFeatures没有学习均值/方差，但仍封装在折内；StandardScaler只学习本训练部分的均值/方差，验证只transform/predict。
- 评价使用不含正则化惩罚项的MAE（MPa）。使用cross_validate时明确scoring='neg_mean_absolute_error'并将负分转换成正MAE；实际键名、输入/输出与shape在代码教学时逐段解释。
- 报告5个验证折的MAE、算术平均值、描述性标准差（本单元预定np.std(ddof=0)），并声明标准差反映该固定划分下的折间波动，不是置信区间或所有未知材料的误差保证。训练折高度重叠，不把5折当成5次独立材料实验。
- 若依据开发CV选定一个方案，先冻结选择、流水线与最终评价步骤，才可用全部开发数据重拟合并一次评价本单元保留测试；不凭测试结果改次数、alpha、划分或种子。本阶段没有开放测试执行。
- learning_curve函数放在理解K折后再实现CV学习曲线；输入为完整Pipeline和原始开发数据，训练量指每折训练部分使用的样本量。函数输出的二维分数数组与fold维度由学生先判断。
- GroupKFold仅讲分组独立性原理；本轮不生成虚构炉次、不完成分组实践。真实炉次或文献来源需有实际元数据并匹配预测目标。

## 证据记录规则

每次代码教学先解释输入、输出、shape和来源。学生先写，教师先审查并提示，再由学生运行保存；没有真实输出就标“未运行”。教师辅助检查与学生运行分开记录；所有图像先由学生独立判断趋势、原因与限制，不预填诊断或候选排名。


## 概念入口Q0（历史；当前实践见下方）

单元08已正确说明横轴为训练样本数和增加数据的诊断用途，尚未实际画过学习曲线。这里检查新的实现步骤，不重学整章。

**教学设计，非实际运行结果：** 模型的多项式次数和Ridge的alpha都保持固定。要画训练样本数分别为12、24、48时的误差点：你会分别用对应数量的训练样本重新fit，还是先用全部训练样本fit一次，再换不同数量的样本计算误差？请说明理由。

Q0首次选第二种错误，已提示后纠正，完整原答与帮助层级保留在四类档案；本题不是单元末正式8题。

教师阶段记录（2026-10-30，不代填下方学生理解）：Q10首次答‘误差没有达到目标要求’，Q11首次答‘趋于平缓’，均为手工synthetic示意图B的正确观察；Q12首次答‘左侧12条，模型训练误差显著低于验证误差，为过拟合’，正确指出图A少样本端的过拟合常见信号与两种误差依据。图A/B都不是实际模型运行，不能仅凭一处误差差距确诊真实模型的原因；不是旧题独立复查或正式8题成绩。


## 我的理解

待学生作答；教师不预填。


## 实践第1步：读取新数据并先封存测试集（提示后修正，已运行）

数据文件：[data/unit09-learning-curves-synthetic.csv](data/unit09-learning-curves-synthetic.csv)。由教师使用NumPy 2.2.6按上方预先固定的default_rng(20261029)配方生成，不重抽种子、不使用单元08数据。150条synthetic data，仅为数学教学设定，不是实验结果。

三列为sample_id、ZrB2_wt_pct、UTS_MPa。sample_id为U09-001至U09-150，只记录行身份，不作为模型特征。ZrB2数值1表示1 wt.%；UTS单位MPa。先生成全部150个Uniform(0,2)输入，再生成150个Normal(0,6)噪声，按465 + 60*x - 24*x**2 + noise构造目标。CSV浮点数以17位有效数字保存，已核验读取后无数值变化；这仅是数据文件检查，不是模型评价。

CSV SHA256：1486c2ffd5758f68fdd79b4d1cd3743e6e7aa6363d5e2dfcc43fbb67a468f165。该文件未改。学生代码格[7]已读取并划分，真实stdout为(120, 3) (30, 3)；尚无模型fit/predict/评分。

原任务：读取CSV为df09；用test_size=0.2、random_state=20261030划分为df09_dev和df09_holdout，仅打印shape。学生首稿[6]传入X09和y09，却只接收两个结果，实际ValueError: too many values to unpack (expected 2)。教师解释一输入返回两份、两输入返回四份，提示完整三列表格；学生先答df09，教师再提示替换输入。修正版已由学生运行为[7]并保存，截图与本地输出一致。记提示后修正，不记首次独立完成；首稿、真实报错和帮助层级保留在四类档案。

输入CSV为150行3列；已保存的实际输出df09_dev.shape=(120,3)、df09_holdout.shape=(30,3)。120条开发数据用于后续曲线/CV，30条holdout保留封存；目前仅展示shape，没有目标预览或评价。sample_id随整行划分，分组ID文件尚未导出。代码格顶部‘当前没有执行’为旧注释，本段与真实计数/输出反映当前状态；教师不替学生改写代码。

下一步只记录分组身份：新建代码格，从df09_dev和df09_holdout各选sample_id一列，分别保存为data/unit09-dev-ids.csv和data/unit09-holdout-ids.csv。使用.to_csv(..., index=False)，其中index=False表示不额外写入pandas的行索引，sample_id列仍保留。输入为已划分的120×3和30×3表，选列后为120×1和30×1，输出预期为相应行数的一列CSV，不是模型分数。先发两行代码检查，暂不运行；不重新划分或展示UTS。

固定两个候选、学习曲线循环、独立读图、5折轮换、Pipeline/cross_validate和评分汇总之后依次进行；本格不提供完整解法。


In [2]:
# 概念解释通过后，由学生逐步编写本单元代码。
# 当前没有执行，也没有模拟模型分数。


import pandas as pd
df09 = pd.read_csv("data/unit09-learning-curves-synthetic.csv")

from sklearn.model_selection import train_test_split
df09_dev, df09_holdout = train_test_split(df09,test_size = 0.2,random_state = 20261030)
print(df09_dev.shape, df09_holdout.shape)

#保存分组的样本编号;table.to_csv("data/example.csv", index=False)、table代表要保存的表格。引号内是输出文件路径，放在括号里。
df09_dev[["sample_id"]].to_csv("data/unit09-dev-ids.csv", index=False)
df09_holdout[["sample_id"]].to_csv("data/unit09-holdout-ids.csv", index=False)

(120, 3) (30, 3)


In [3]:
#划分dev
df09_train_pool, df09_val = train_test_split(df09_dev,test_size = 0.2,random_state = 20261031)
print(df09_train_pool.shape, df09_val.shape)

X09_pool = df09_train_pool[["ZrB2_wt_pct"]]
y09_pool = df09_train_pool[["UTS_MPa"]]
X09_val = df09_val[["ZrB2_wt_pct"]]
y09_val = df09_val[["UTS_MPa"]]
print(X09_pool.shape, y09_pool.shape, X09_val.shape, y09_val.shape)

X09_12 = X09_pool.iloc[:12]  #取前12行
y09_12 = y09_pool.iloc[:12]
print(X09_12.shape, y09_12.shape)

(96, 3) (24, 3)
(96, 1) (96, 1) (24, 1) (24, 1)
(12, 1) (12, 1)


In [11]:
# 训练第一个模型
from sklearn.preprocessing import StandardScaler
scaler09_12 = StandardScaler()#创建标准化器

from sklearn.linear_model import LinearRegression 
reg09_12 = LinearRegression()#创建回归器

from sklearn.pipeline import Pipeline #引入管道：多项式展开，标准化，回归模型("步骤名称", 工具对象)
pipe09_linear12 = Pipeline([
    ("scale", scaler09_12),
    ("reg", reg09_12),
])
pipe09_linear12.fit(X09_12, y09_12)


# 预测
y09_12_pred = pipe09_linear12.predict(X09_12)
y09_val_pred = pipe09_linear12.predict(X09_val)
print(y09_12_pred.shape, y09_val_pred.shape)

# 计算MAE
from sklearn.metrics import mean_absolute_error#导入计算MAE的函数
mae09_train12 = mean_absolute_error(y09_12,y09_12_pred)#结果变量 = 函数名称(真实值, 预测值)
print(f"训练MAE:  {mae09_train12:.3f} MPa")#f让花括号中的变量插入文本，:.3f 表示显示小数点后三位。这只改变显示，原变量的数值不变。
mae09_val12 = mean_absolute_error(y09_val,y09_val_pred)#计算验证集MAE
print(f"验证MAE: {mae09_val12:.3f} MPa")

# 得到横坐标为24的点
X09_24 = X09_pool.iloc[:24]  #取前24行
y09_24 = y09_pool.iloc[:24]
print(X09_24.shape, y09_24.shape)

scaler09_24 = StandardScaler()#创建标准化器
reg09_24 = LinearRegression()#创建回归器
pipe09_linear24 = Pipeline([
    ("scale", scaler09_24),
    ("reg", reg09_24),
])
pipe09_linear24.fit(X09_24, y09_24)

(12, 1) (24, 1)
训练MAE:  5.838 MPa
验证MAE: 7.250 MPa
(24, 1) (24, 1)


,steps,"[('scale', ...), ('reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None


# 结课评估（尚未开始）

完成教学与实践后逐题进行3道概念题、2道数学理解题、2道Python题、1道材料迁移题。首次错误不得由后续正确答案覆盖。正式题0/8作答，0/8验收；本Notebook仍为学习中。


## 来源与使用限制

- 仓库课程：markdown/week6.md 10.3—10.6；code/ex5-bias vs variance/ML-Exercise5.ipynb。
- 课程10.3的“交叉验证集”主要指固定验证集，不等同K-fold。10.3测试误差式误用了cv下标，按test样本解释。
- 原Ex5分别对X/Xval/Xtest计算缩放统计量，且末尾用测试误差选lambda；不把这些步骤作为本单元的验证范例。原文件保留不动。
- [scikit-learn学习曲线](https://scikit-learn.org/stable/modules/learning_curve.html)
- [交叉验证与分组划分](https://scikit-learn.org/stable/modules/cross_validation.html)
- [评分约定](https://scikit-learn.org/stable/modules/model_evaluation.html#scoring-parameter)
- [Pipeline与数据泄漏](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)

数据来自数学教学设定，任何CV分数均不能写成真实材料实验性能或全部未知材料的预测保证。
